In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPool2D, BatchNormalization, Input, Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
data_path = '/content/drive/MyDrive/soybean'

# Function to load and preprocess images
def load_data(dataset_path, mode='train', img_size=(150, 150)):
    images, labels = [], []
    categories = ['Broken soybeans', 'Immature soybeans', 'Intact soybeans', 'Skin-damaged soybeans', 'Spotted soybeans']

    for label, category in enumerate(categories):
        category_path = os.path.join(dataset_path, mode, category)
        for img_file in os.listdir(category_path):
            img_path = os.path.join(category_path, img_file)
            image = cv2.imread(img_path)
            image = cv2.resize(image, img_size)
            images.append(image)
            labels.append(label)

    images = np.array(images).astype('float32') / 255.0  # Normalize
    labels = to_categorical(labels, num_classes=len(categories))  # One-hot encode labels
    return images, labels

In [ ]:
# Load training and testing data
train_images, train_labels = load_data(data_path, mode='train')
test_images, test_labels = load_data(data_path, mode='test')

In [ ]:
# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.3,
    shear_range=0.2
)
datagen.fit(train_images)

In [ ]:
# Define Improved CNN Model
i = Input(shape=(150, 150, 3))

# Block 1
c1 = Conv2D(32, (3,3), activation='relu', padding='same')(i)
c1 = BatchNormalization()(c1)
c1 = Conv2D(32, (3,3), activation='relu', padding='same')(c1)
c1 = MaxPool2D(pool_size=(2,2))(c1)

# Block 2
c2 = Conv2D(64, (3,3), activation='relu', padding='same')(c1)
c2 = BatchNormalization()(c2)
c2 = Conv2D(64, (3,3), activation='relu', padding='same')(c2)
c2 = MaxPool2D(pool_size=(2,2))(c2)

# Block 3
c3 = Conv2D(128, (3,3), activation='relu', padding='same')(c2)
c3 = BatchNormalization()(c3)
c3 = Conv2D(128, (3,3), activation='relu', padding='same')(c3)
c3 = MaxPool2D(pool_size=(2,2))(c3)

# Block 4
c4 = Conv2D(256, (3,3), activation='relu', padding='same')(c3)
c4 = BatchNormalization()(c4)
c4 = Conv2D(256, (3,3), activation='relu', padding='same')(c4)
c4 = MaxPool2D(pool_size=(2,2))(c4)

# Fully Connected Layers
f1 = Flatten()(c4)
d1 = Dense(512, activation='relu')(f1)
d1 = Dropout(0.5)(d1)  # Prevent overfitting
d2 = Dense(256, activation='relu')(d1)
d2 = Dropout(0.5)(d2)
o = Dense(5, activation='softmax')(d2)  # 5 output classes

In [ ]:
# Compile model
model = Model(i, o)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# Train model
h = model.fit(datagen.flow(train_images, train_labels, batch_size=32),
              epochs=35,
              validation_data=(test_images, test_labels))

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/35
138/138 ━━━━━━━━━━━━━━━━━━━━ 58s 293ms/step - accuracy: 0.2902 - loss: 2.2826 - val_accuracy: 0.2181 - val_loss: 1.7194
Epoch 2/35
138/138 ━━━━━━━━━━━━━━━━━━━━ 54s 183ms/step - accuracy: 0.4284 - loss: 1.3705 - val_accuracy: 0.2036 - val_loss: 2.6105
Epoch 3/35
138/138 ━━━━━━━━━━━━━━━━━━━━ 27s 193ms/step - accuracy: 0.4808 - loss: 1.2572 - val_accuracy: 0.2081 - val_loss: 2.4044
Epoch 4/35
138/138 ━━━━━━━━━━━━━━━━━━━━ 25s 183ms/step - accuracy: 0.5508 - loss: 1.1703 - val_accuracy: 0.5412 - val_loss: 1.2771
Epoch 5/35
138/138 ━━━━━━━━━━━━━━━━━━━━ 25s 184ms/step - accuracy: 0.5792 - loss: 1.0826 - val_accuracy: 0.6534 - val_loss: 0.8902
Epoch 6/35
138/138 ━━━━━━━━━━━━━━━━━━━━ 26s 189ms/step - accuracy: 0.6382 - loss: 0.9870 - val_accuracy: 0.7041 - val_loss: 0.8308
Epoch 7/35
138/138 ━━━━━━━━━━━━━━━━━━━━ 26s 190ms/step - accuracy: 0.6483 - loss: 0.9626 - val_accuracy: 0.7701 - val_loss: 0.7104
Epoch 8/35
138/138 ━━━━━━━━━━━━━━━━━━━━ 26s 187ms/step - accuracy: 0.6861 - loss: 0

In [ ]:
# Evaluate model
r = model.predict(test_images)
rounded_preds = np.argmax(r, axis=1)
true_labels = np.argmax(test_labels, axis=1)

35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step


In [ ]:
print(classification_report(true_labels, rounded_preds))

              precision    recall  f1-score   support

           0       0.80      0.78      0.79       201
           1       0.95      0.85      0.90       225
           2       0.80      0.97      0.87       241
           3       0.90      0.94      0.92       226
           4       0.92      0.78      0.84       212

    accuracy                           0.87      1105
   macro avg       0.87      0.86      0.86      1105
weighted avg       0.87      0.87      0.87      1105



In [ ]:
print(confusion_matrix(true_labels, rounded_preds))

[[156   4   9  20  12]
 [  1 191  30   2   1]
 [  2   3 233   2   1]
 [  6   1   7 212   0]
 [ 31   2  14   0 165]]
